In [0]:
dbutils.library.restartPython()

In [0]:
from openai import AzureOpenAI

client = AzureOpenAI(
api_key = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-KEY")
endpoint = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-ENDPOINT")
api_version = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-API-VERSION")
)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Explain what fraud detection means in banking"}
    ]
)

print(response.choices[0].message.content)

In [0]:
transactions_df = spark.sql("""
SELECT transaction_id,
       customer_id,
       amount,
       location,
       merchant
FROM fraud.transactions
LIMIT 10
""")

transactions_df.display()

In [0]:
transactions_text = "\n".join(
    [
        f"Transaction {r.transaction_id}: customer {r.customer_id}, amount {r.amount}, location {r.location}, merchant {r.merchant}"
        for r in transactions_df.collect()
    ]
)

print(transactions_text)

In [0]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a fraud investigation analyst."
        },
        {
            "role": "user",
            "content": f"""
Investigate these suspicious transactions:

{transactions_text}

Explain:
- suspicious patterns
- high risk transactions
- recommended actions
"""
        }
    ]
)

investigation_report = response.choices[0].message.content

print(investigation_report)

In [0]:
from openai import AzureOpenAI

client = AzureOpenAI(
api_key = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-KEY")
endpoint = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-ENDPOINT")
api_version = dbutils.secrets.get("ai-secrets", "AZURE-OPENAI-API-VERSION")
)

In [0]:
risk_agent_prompt = "You are a financial fraud risk analysis agent."

alert_agent_prompt = "You are responsible for generating fraud alerts based on risk analysis."

investigation_agent_prompt = "You are a senior fraud investigator who writes detailed investigation reports."

In [0]:
transactions_df = spark.sql("""
SELECT transaction_id,
       customer_id,
       amount,
       location,
       merchant
FROM fraud.transactions
LIMIT 10
""")

transactions_df.display()

In [0]:
transactions_text = "\n".join(
    [
        f"Transaction {r.transaction_id}: customer {r.customer_id}, amount {r.amount}, location {r.location}, merchant {r.merchant}"
        for r in transactions_df.collect()
    ]
)

In [0]:
risk_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": risk_agent_prompt},
        {"role": "user", "content": f"Analyze these transactions for fraud risk:\n{transactions_text}"}
    ]
)

risk_analysis = risk_response.choices[0].message.content

print(risk_analysis)

In [0]:
alert_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": alert_agent_prompt},
        {"role": "user", "content": f"Generate fraud alerts based on this risk analysis:\n{risk_analysis}"}
    ]
)

fraud_alerts = alert_response.choices[0].message.content

print(fraud_alerts)

In [0]:
investigation_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": investigation_agent_prompt},
        {"role": "user", "content": f"Investigate these fraud alerts:\n{fraud_alerts}"}
    ]
)

investigation_report = investigation_response.choices[0].message.content

print(investigation_report)

In [0]:
report_df = spark.createDataFrame(
    [(investigation_report,)],
    ["fraud_report"]
)

report_df.write.format("delta").mode("overwrite").saveAsTable(
    "fraud.fraud_investigation_report"
)

In [0]:
spark.sql("""
SELECT *
FROM fraud.fraud_investigation_report
""").display()